In [66]:
# Instal library yang dipakai (lxml = parser HTML, beautifulsoup4 = scraping manual)
!pip -q install lxml beautifulsoup4

import os, re, json, time, hashlib, io, csv
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# data/raw = data mentah (TIDAK diubah), data/processed = hasil olahan, docs = metadata & bukti
for d in ["data/raw", "data/processed", "docs"]:
    os.makedirs(d, exist_ok=True)

In [67]:
EMAIL = "wiraastinarasmawan@gmail.com"
TOPIK = "Destinasi Wisata Indonesia"

# Ketentuan soal: User-Agent berisi email kampus
assert "@" in EMAIL and "ISI_EMAIL" not in EMAIL, "Isi EMAIL dengan alamat email aktif!"
USER_AGENT = f"INSTIKI-BIFP243-UTS/1.0 ({EMAIL})"
HEADERS = {"User-Agent": USER_AGENT}
TIMEOUT = 20   # detik; wajib di setiap request
DELAY = 1.5    # jeda antar-request (detik)
print(USER_AGENT)

INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawan@gmail.com)


In [68]:
# Hash MD5 untuk bukti integritas berkas mentah
def md5_file(path, chunk=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

# Catatan provenance: setiap sumber dicatat (URL, tanggal akses, lisensi, MD5)
PROVENANCE = []
def catat_provenance(nama, metode, url, path, lisensi, catatan=""):
    PROVENANCE.append({
        "sumber": nama, "metode_akuisisi": metode, "url": url,
        "tanggal_akses_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "lisensi": lisensi, "berkas_mentah": path,
        "ukuran_byte": os.path.getsize(path), "md5": md5_file(path), "catatan": catatan,
    })
    print(f"✔ provenance: {nama} | md5={PROVENANCE[-1]['md5']}")

# Request sopan: timeout + User-Agent + jeda + penanganan status code + retry backoff
def polite_get(url, params=None, retries=3, backoff=2.0):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(DELAY)                       # jeda antar-request
                return r
            if r.status_code == 429:                    # terlalu banyak request: patuhi Retry-After
                wait = int(r.headers.get("Retry-After", backoff ** attempt))
                print(f"429, tunggu {wait}s"); time.sleep(wait)
            elif 500 <= r.status_code < 600:            # error server: retry dengan backoff
                print(f"{r.status_code}, retry {attempt}/{retries}"); time.sleep(backoff ** attempt)
            else:                                       # 4xx lain: kesalahan permintaan, tidak di-retry
                r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"Gagal koneksi ({type(e).__name__}), retry {attempt}/{retries}")
            time.sleep(backoff ** attempt)
    raise RuntimeError(f"Gagal mengambil {url} setelah {retries} percobaan")

In [69]:
# Hash MD5 untuk bukti integritas berkas mentah
def md5_file(path, chunk=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

# Catatan provenance: setiap sumber dicatat (URL, tanggal akses, lisensi, MD5)
PROVENANCE = []
def catat_provenance(nama, metode, url, path, lisensi, catatan=""):
    PROVENANCE.append({
        "sumber": nama, "metode_akuisisi": metode, "url": url,
        "tanggal_akses_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "lisensi": lisensi, "berkas_mentah": path,
        "ukuran_byte": os.path.getsize(path), "md5": md5_file(path), "catatan": catatan,
    })
    print(f"✔ provenance: {nama} | md5={PROVENANCE[-1]['md5']}")

# Request sopan: timeout + User-Agent + jeda + penanganan status code + retry backoff
def polite_get(url, params=None, retries=3, backoff=2.0):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(DELAY)                       # jeda antar-request
                return r
            if r.status_code == 429:                    # terlalu banyak request: patuhi Retry-After
                wait = int(r.headers.get("Retry-After", backoff ** attempt))
                print(f"429, tunggu {wait}s"); time.sleep(wait)
            elif 500 <= r.status_code < 600:            # error server: retry dengan backoff
                print(f"{r.status_code}, retry {attempt}/{retries}"); time.sleep(backoff ** attempt)
            else:                                       # 4xx lain: kesalahan permintaan, tidak di-retry
                r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"Gagal koneksi ({type(e).__name__}), retry {attempt}/{retries}")
            time.sleep(backoff ** attempt)
    raise RuntimeError(f"Gagal mengambil {url} setelah {retries} percobaan")

In [70]:
# ==================================================
# KONFIGURASI SUMBER BERKAS DAN HASIL PENGECEKAN
# DATA CARD KAGGLE
# ==================================================

FILE_PATH = "data/raw/tourism_with_id.csv"

FILE_URL = (
    "https://www.kaggle.com/datasets/"
    "aprabowo/indonesia-tourism-destination"
)

URL_KAGGLE_TOS = "https://www.kaggle.com/terms"

# ==================================================
# INFORMASI LISENSI KAGGLE
# ==================================================

LISENSI_KAGGLE = (
    "Data files © Original Authors. "
    "Pada bagian Acknowledgements, pengunggah menyatakan "
    "'ALL LICENSE ARE FREE TO USE DATASET'. "
    "Namun, lisensi formal masih perlu diverifikasi."
)

# ==================================================
# INFORMASI ASAL DATA
# ==================================================

ASAL_DATA_KAGGLE = (
    "Dataset diunggah oleh A_Prabowo dan dikaitkan dengan "
    "Capstone Project Bangkit Academy 2021 bernama GetLoc. "
    "Data Card memberikan kredit kepada GetLoc Team. "
    "Sumber asli data destinasi wisata tidak dijelaskan "
    "secara rinci pada Data Card."
)

# None = belum dapat dipastikan
ASAL_BUKAN_PLATFORM_TERLARANG = None

# False = belum dikonfirmasi sudah membaca ketentuan
SUDAH_BACA_TOS = False

# ==================================================
# PENGATURAN PENGUNDUHAN
# ==================================================

PAKAI_KAGGLE_API = False

print("Konfigurasi Kaggle berhasil diisi.")
print("File:", FILE_PATH)
print("URL:", FILE_URL)
print("Lisensi:", LISENSI_KAGGLE)
print("Asal data:", ASAL_DATA_KAGGLE)
print("Asal bukan platform terlarang:", ASAL_BUKAN_PLATFORM_TERLARANG)
print("Sudah membaca TOS:", SUDAH_BACA_TOS)

Konfigurasi Kaggle berhasil diisi.
File: data/raw/tourism_with_id.csv
URL: https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination
Lisensi: Data files © Original Authors. Pada bagian Acknowledgements, pengunggah menyatakan 'ALL LICENSE ARE FREE TO USE DATASET'. Namun, lisensi formal masih perlu diverifikasi.
Asal data: Dataset diunggah oleh A_Prabowo dan dikaitkan dengan Capstone Project Bangkit Academy 2021 bernama GetLoc. Data Card memberikan kredit kepada GetLoc Team. Sumber asli data destinasi wisata tidak dijelaskan secara rinci pada Data Card.
Asal bukan platform terlarang: None
Sudah membaca TOS: False


In [71]:
# (OPSIONAL) Kaggle API resmi: kredensial dari Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY), tidak ditulis di notebook
if PAKAI_KAGGLE_API and not os.path.exists(FILE_PATH):
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !pip -q install kaggle
    # hanya unduh satu berkas yang dibutuhkan (berkas lain berisi data pengguna dummy)
    !kaggle datasets download -d aprabowo/indonesia-tourism-destination -f tourism_with_id.csv -p data/raw --unzip

# Jika berkas belum ada, minta upload manual
if not os.path.exists(FILE_PATH):
    from google.colab import files
    up = files.upload()
    for nama, isi in up.items():
        with open(f"data/raw/{nama}", "wb") as f:
            f.write(isi)
assert os.path.exists(FILE_PATH), "Berkas dataset belum ada di data/raw"
print("Berkas siap:", FILE_PATH, f"({os.path.getsize(FILE_PATH)/1024:.1f} KB)")

Berkas siap: data/raw/tourism_with_id.csv (357.2 KB)


In [72]:
# Deteksi encoding dan delimiter supaya parameter read_csv tepat
raw = open(FILE_PATH, "rb").read(20000)
for enc in ["utf-8", "utf-8-sig", "latin-1"]:
    try:
        sample = raw.decode(enc); ENC = enc; break
    except UnicodeDecodeError:
        continue
try:
    SEP = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
except Exception:
    SEP = ","
print("encoding:", ENC, "| sep:", repr(SEP))
print(sample[:300])   # intip isi awal

encoding: utf-8 | sep: ','
Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,,
1,Monumen Nasional,"Monumen Nasional atau yang populer disingkat dengan Monas atau Tugu Monas adalah monumen peringatan setinggi 132 meter (433 kaki) yang didirikan untuk mengenang perlawanan dan perjuanga


In [73]:
# Baca berkas dengan parameter yang tepat (sep, encoding, na_values, dtype, skiprows)
df_file = pd.read_csv(
    FILE_PATH,
    sep=SEP,
    encoding=ENC,
    skiprows=0,                                   # tidak ada baris judul/catatan di atas header
    na_values=["-", "--", "NA", "N/A", ""],       # penanda nilai kosong
    dtype={"Place_Id": "int64", "Price": "int64"},  # paksa tipe untuk ID dan harga
)
df_file.head()

,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Unnamed: 11,Unnamed: 12
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,NaN,1
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,NaN,2
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,NaN,3
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,NaN,4
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,NaN,5


In [74]:
# Identifikasi dataset: ukuran, tipe data, nilai kosong, cakupan wilayah/waktu
n_baris, n_kolom = df_file.shape
ukuran_byte = os.path.getsize(FILE_PATH)
print(f"Baris: {n_baris} | Kolom: {n_kolom} | Ukuran berkas: {ukuran_byte/1024:.1f} KB")
assert n_baris >= 100, "Dataset harus minimal 100 baris!"

info = pd.DataFrame({
    "tipe_data": df_file.dtypes.astype(str),
    "n_kosong": df_file.isna().sum(),
    "n_unik": df_file.nunique(),
    "contoh_nilai": df_file.iloc[0].astype(str).str.slice(0, 40),
})
display(info)

# Kolom sisa tanpa nama jelas (artefak ekspor); dicatat, tidak diubah di data mentah
kolom_sisa = [c for c in df_file.columns if str(c).startswith("Unnamed")]
print("Kolom tanpa nama (artefak):", kolom_sisa)
print("\nCakupan wilayah (jumlah destinasi per kota):")
print(df_file["City"].value_counts().to_string())
print("\nKategori:", df_file["Category"].value_counts().to_dict())
print("\nRentang nilai numerik:"); print(df_file[["Price", "Rating", "Time_Minutes", "Lat", "Long"]].describe().loc[["min", "max"]].to_string())
print("Kolom bertipe tanggal:", [c for c in df_file.columns if "date" in c.lower() or "tahun" in c.lower()] or "tidak ada")

Baris: 437 | Kolom: 13 | Ukuran berkas: 357.2 KB


,tipe_data,n_kosong,n_unik,contoh_nilai
Place_Id,int64,0,437,1
Place_Name,object,0,437,Monumen Nasional
Description,object,0,437,Monumen Nasional atau yang populer disin
Category,object,0,6,Budaya
City,object,0,5,Jakarta
Price,int64,0,50,20000
Rating,float64,0,14,4.6
Time_Minutes,float64,232,15,15.0
Coordinate,object,0,437,"{'lat': -6.1753924, 'lng': 106.8271528}"
Lat,float64,0,437,-6.1753924


Kolom tanpa nama (artefak): ['Unnamed: 11', 'Unnamed: 12']

Cakupan wilayah (jumlah destinasi per kota):
City
Yogyakarta    126
Bandung       124
Jakarta        84
Semarang       57
Surabaya       46

Kategori: {'Taman Hiburan': 135, 'Budaya': 117, 'Cagar Alam': 106, 'Bahari': 47, 'Tempat Ibadah': 17, 'Pusat Perbelanjaan': 15}

Rentang nilai numerik:
        Price  Rating  Time_Minutes       Lat        Long
min       0.0     3.4          10.0 -8.197894  103.931398
max  900000.0     5.0         360.0  1.078880  112.821662
Kolom bertipe tanggal: tidak ada


In [75]:
# Catat provenance berkas (lisensi diambil dari isian Data Card di atas)
catat_provenance("Kaggle - Indonesia Tourism Destination",
                 "Kaggle API" if PAKAI_KAGGLE_API else "unduh manual dari Kaggle",
                 FILE_URL, FILE_PATH, LISENSI_KAGGLE,
                 f"{n_baris} baris x {n_kolom} kolom; asal data: {ASAL_DATA_KAGGLE}")

✔ provenance: Kaggle - Indonesia Tourism Destination | md5=722ca7f02aa2f847c41798c3f6a5d7a5


In [76]:
# Estimasi storage 1 tahun = ukuran berkas x 12 bulan + cadangan 30%
# ASUMSI: berkas diunduh ulang setiap bulan dan disimpan sebagai berkas baru tanpa kompresi;
# cadangan 30% untuk hasil olahan (merge), backup, dan pertumbuhan jumlah destinasi.
ukuran_mb = ukuran_byte / (1024 ** 2)
bulanan = ukuran_mb * 12
cadangan = bulanan * 0.30
total_mb = bulanan + cadangan
print(f"Ukuran berkas      : {ukuran_mb:.4f} MB")
print(f"x 12 bulan         : {bulanan:.4f} MB")
print(f"+ cadangan 30%     : {cadangan:.4f} MB")
print(f"TOTAL estimasi/thn : {total_mb:.4f} MB (= {total_mb/1024:.6f} GB)")
ESTIMASI_STORAGE_MB = round(total_mb, 4)

Ukuran berkas      : 0.3489 MB
x 12 bulan         : 4.1862 MB
+ cadangan 30%     : 1.2559 MB
TOTAL estimasi/thn : 5.4421 MB (= 0.005315 GB)


In [77]:
# Ringkasan lisensi dan ketentuan tiap sumber (bukti Soal 2)
LISENSI_SUMBER = pd.DataFrame([
    ["Kaggle - Indonesia Tourism Destination", LISENSI_KAGGLE, FILE_URL, URL_KAGGLE_TOS],
    ["Open-Meteo Historical API", "CC BY 4.0 (atribusi wajib; non-komersial)", "https://open-meteo.com/en/license", "https://open-meteo.com/en/terms"],
    ["Wikipedia (en)", "CC BY-SA 4.0 (atribusi + share-alike)", "https://en.wikipedia.org/wiki/Wikipedia:Text_of_the_Creative_Commons_Attribution-ShareAlike_4.0_International_License", "https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use"],
], columns=["sumber", "lisensi", "url_lisensi", "url_ketentuan"])
display(LISENSI_SUMBER)

,sumber,lisensi,url_lisensi,url_ketentuan
0,Kaggle - Indonesia Tourism Destination,Data files © Original Authors. Pada bagian Ack...,https://www.kaggle.com/datasets/aprabowo/indon...,https://www.kaggle.com/terms
1,Open-Meteo Historical API,CC BY 4.0 (atribusi wajib; non-komersial),https://open-meteo.com/en/license,https://open-meteo.com/en/terms
2,Wikipedia (en),CC BY-SA 4.0 (atribusi + share-alike),https://en.wikipedia.org/wiki/Wikipedia:Text_o...,https://foundation.wikimedia.org/wiki/Policy:T...


In [78]:
# Deteksi kandidat PII dan quasi-identifier (heuristik nama kolom + pola nilai)
pola_kolom = {
    "PII langsung": r"^(nama|name|email|telp|phone|hp|nik|ktp|alamat|address|npwp|paspor)$|user",
    "Quasi-identifier": r"city|kota|lat|long|coordinate|koordinat|kecamatan|desa|usia|age|gender|kebangsaan",
}
temuan = []
for kol in df_file.columns:
    for jenis, pola in pola_kolom.items():
        if re.search(pola, str(kol), flags=re.I):
            temuan.append({"kolom": kol, "kategori": jenis})

# Pola nilai pada kolom teks: email, telepon Indonesia, NIK 16 digit
pola_nilai = {"email": r"[\w.\-]+@[\w.\-]+\.\w+", "telepon": r"(?:\+62|\b08)\d{8,12}", "nik_16_digit": r"\b\d{16}\b"}
kolom_teks = df_file.select_dtypes(include="object").columns
for kol in kolom_teks:
    for nama, p in pola_nilai.items():
        n = df_file[kol].astype(str).str.contains(p, regex=True).sum()
        if n:
            temuan.append({"kolom": kol, "kategori": f"PII (pola nilai {nama}, {n} baris)"})

pii_df = pd.DataFrame(temuan, columns=["kolom", "kategori"])
display(pii_df)
PII_LANGSUNG = pii_df[pii_df["kategori"].str.startswith("PII")]
QUASI = pii_df[pii_df["kategori"] == "Quasi-identifier"]["kolom"].tolist()
print("PII langsung ditemukan:", len(PII_LANGSUNG), "| quasi-identifier:", QUASI)

,kolom,kategori
0,City,Quasi-identifier
1,Coordinate,Quasi-identifier
2,Lat,Quasi-identifier
3,Long,Quasi-identifier


PII langsung ditemukan: 0 | quasi-identifier: ['City', 'Coordinate', 'Lat', 'Long']


In [79]:
# Lokasi = 5 kota pada dataset destinasi; rentang tanggal = tahun 2024
LOKASI = {
    "Jakarta": (-6.2088, 106.8456),
    "Yogyakarta": (-7.7956, 110.3695),
    "Bandung": (-6.9175, 107.6191),
    "Semarang": (-6.9667, 110.4167),
    "Surabaya": (-7.2575, 112.7521),
}
TGL_MULAI, TGL_AKHIR = "2024-01-01", "2024-12-31"
URL_API = "https://archive-api.open-meteo.com/v1/archive"

frames = []
for kota, (lat, lon) in LOKASI.items():
    params = {"latitude": lat, "longitude": lon, "start_date": TGL_MULAI, "end_date": TGL_AKHIR,
              "daily": "temperature_2m_mean,precipitation_sum,sunshine_duration",
              "timezone": "Asia/Jakarta"}
    r = polite_get(URL_API, params=params)       # timeout, User-Agent, jeda, retry sudah di fungsi
    js = r.json()
    # simpan respons JSON mentah apa adanya
    p_raw = f"data/raw/openmeteo_{kota}.json"
    with open(p_raw, "w") as f:
        json.dump(js, f)
    catat_provenance(f"Open-Meteo Historical API - {kota}", "API", r.url, p_raw,
                     "CC BY 4.0", "cuaca harian historis 2024; atribusi Open-Meteo")
    # normalisasi JSON (list paralel di kunci 'daily') menjadi tabel
    d = pd.DataFrame(js["daily"])
    d["lokasi"] = kota
    d["latitude"], d["longitude"] = js["latitude"], js["longitude"]
    frames.append(d)

df_api = pd.concat(frames, ignore_index=True).rename(columns={"time": "tanggal"})
df_api["tanggal"] = pd.to_datetime(df_api["tanggal"])
assert len(df_api) >= 30, "Tabel API harus minimal 30 baris"
df_api.to_csv("data/processed/data_api_cuaca.csv", index=False)
print(df_api.shape)
df_api.head()

✔ provenance: Open-Meteo Historical API - Jakarta | md5=5947251f4e5024368641b3a3efa6fb64
✔ provenance: Open-Meteo Historical API - Yogyakarta | md5=564017edb8037adb5cb76e4ec8c630a5
✔ provenance: Open-Meteo Historical API - Bandung | md5=ff7469b63c5992f823ed7b2fd51b89a7
✔ provenance: Open-Meteo Historical API - Semarang | md5=0daa501d04685d256c28d552d7732480
✔ provenance: Open-Meteo Historical API - Surabaya | md5=071270d667cfb62c6240a7313b38bd15
(1830, 7)


,tanggal,temperature_2m_mean,precipitation_sum,sunshine_duration,lokasi,latitude,longitude
0,2024-01-01,26.8,11.0,26267.57,Jakarta,-6.221441,106.856186
1,2024-01-02,27.4,8.4,27591.26,Jakarta,-6.221441,106.856186
2,2024-01-03,25.8,29.5,8244.02,Jakarta,-6.221441,106.856186
3,2024-01-04,25.2,59.8,4901.31,Jakarta,-6.221441,106.856186
4,2024-01-05,26.6,15.9,39525.22,Jakarta,-6.221441,106.856186


In [80]:
# Bukti penanganan status code: permintaan sengaja salah (latitude di luar rentang) -> 4xx
try:
    polite_get(URL_API, params={"latitude": 999, "longitude": 0, "start_date": TGL_MULAI,
                                "end_date": TGL_AKHIR, "daily": "temperature_2m_mean"})
except requests.HTTPError as e:
    print("Status code error ditangani:", e.response.status_code)
except RuntimeError as e:
    print("Gagal setelah retry:", e)

Status code error ditangani: 400


In [81]:
# ✏️ Halaman target dan pencocok tabel
URL_HALAMAN = "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population"

def cek_kepatuhan(url, user_agent=USER_AGENT):
    # robots.txt selalu berada di root domain, bukan di belakang path halaman
    p = urlparse(url)
    robots_url = f"{p.scheme}://{p.netloc}/robots.txt"
    r = polite_get(robots_url)                     # diambil dengan User-Agent kita
    rp = RobotFileParser()
    rp.parse(r.text.splitlines())
    return {
        "halaman": url, "robots_url": robots_url, "status_code_robots": r.status_code,
        "user_agent": user_agent,
        "boleh_diakses": rp.can_fetch(user_agent, url),
        "crawl_delay_ua_kita": rp.crawl_delay(user_agent),
        "waktu_cek_utc": datetime.now(timezone.utc).isoformat(),
    }, r.text

hasil_robots, isi_robots = cek_kepatuhan(URL_HALAMAN)
Path("data/raw/robots_txt_evidence.txt").write_text(isi_robots, encoding="utf-8")
Path("docs/robots_check.json").write_text(json.dumps(hasil_robots, indent=2), encoding="utf-8")
print(json.dumps(hasil_robots, indent=2))

# Hentikan jika tidak diizinkan; jika ada crawl-delay untuk UA kita, pakai sebagai jeda
assert hasil_robots["boleh_diakses"], "Dilarang scraping: robots.txt tidak mengizinkan halaman ini"
if hasil_robots["crawl_delay_ua_kita"]:
    DELAY = max(DELAY, hasil_robots["crawl_delay_ua_kita"])
catat_provenance("Wikipedia robots.txt", "HTTP GET", hasil_robots["robots_url"],
                 "data/raw/robots_txt_evidence.txt", "n/a (bukti pemeriksaan kepatuhan)")

{
  "halaman": "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population",
  "robots_url": "https://en.wikipedia.org/robots.txt",
  "status_code_robots": 200,
  "user_agent": "INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawan@gmail.com)",
  "boleh_diakses": true,
  "crawl_delay_ua_kita": null,
  "waktu_cek_utc": "2026-11-18T10:12:38.422896+00:00"
}
✔ provenance: Wikipedia robots.txt | md5=fc15d80fd6a4afd9f22975e740c1b48c


In [82]:
# Ambil halaman (timeout, UA, jeda, cek status ada di polite_get) dan simpan HTML mentah
r = polite_get(URL_HALAMAN)
Path("data/raw/halaman_wikipedia.html").write_text(r.text, encoding="utf-8")
catat_provenance("Wikipedia - List of Indonesian cities by population", "web scraping statis",
                 URL_HALAMAN, "data/raw/halaman_wikipedia.html", "CC BY-SA 4.0",
                 "HTML mentah; atribusi Wikipedia")

soup = BeautifulSoup(r.text, "lxml")                # lxml: cepat dan toleran
print("Judul halaman :", soup.title.string.strip())
print("Jumlah <table>:", len(soup.select("table")))

✔ provenance: Wikipedia - List of Indonesian cities by population | md5=7024fa5072368502909e684c79cf42ca
Judul halaman : List of Indonesian cities by population - Wikipedia
Jumlah <table>: 9


In [83]:
# Ekstraksi tabel: cara cepat (pandas.read_html) vs cara manual (BeautifulSoup + selector)
# Tabel dipilih lewat nama kolom ("City" dan "Province"), bukan nomor urut (lebih tahan perubahan)
kandidat = pd.read_html(io.StringIO(r.text), match="Province")
df_cepat = next(t for t in kandidat if {"City", "Province"} <= set(map(str, t.columns)))

tabel_manual = None
for t in soup.select("table.wikitable"):
    header = [th.get_text(strip=True) for th in t.select("tr")[0].select("th")]
    if "City" in header and "Province" in header:
        tabel_manual = t; break
assert tabel_manual is not None, "Tabel target tidak ditemukan"
rows = [[re.sub(r"\[.*?\]", "", c.get_text(" ", strip=True)) for c in tr.select("th, td")]
        for tr in tabel_manual.select("tr")[1:]]
df_manual = pd.DataFrame(rows)

print("read_html:", df_cepat.shape, "| manual:", df_manual.shape)
assert df_cepat.shape[0] == df_manual.shape[0], "Jumlah baris cara cepat dan manual berbeda"

# Bersihkan penanda catatan kaki seperti [3] dari teks dan rapikan nama kolom
df_scrape = df_cepat.copy()
for kol in ["City", "Province", "Region"]:
    df_scrape[kol] = df_scrape[kol].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip()
df_scrape = df_scrape.rename(columns={"2020 census [2]": "Population_2020"})[["City", "Province", "Region", "Population_2020"]]
df_scrape.to_csv("data/processed/tabel_wikipedia_kota.csv", index=False)
display(df_scrape.head(10))

read_html: (95, 8) | manual: (95, 8)


,City,Province,Region,Population_2020
0,Jakarta,Special Capital Region of Jakarta,Java,10562088
1,Surabaya,East Java,Java,2874314
2,Bekasi,West Java,Java,2543676
3,Bandung,West Java,Java,2444160
4,Medan,North Sumatra,Sumatra,2435252
5,Depok,West Java,Java,2056335
6,Tangerang,Banten,Java,1895486
7,Palembang,South Sumatra,Sumatra,1668848
8,Semarang,Central Java,Java,1653524
9,Makassar,South Sulawesi,Sulawesi,1423877


In [84]:
# Uji apakah konten ada di HTML awal (jika tidak, halaman dirender JavaScript dan perlu Selenium/Playwright)
ada_di_html = all(k in r.text for k in ["Surabaya", "Yogyakarta"])
print("Konten ada di HTML awal:", ada_di_html,
      "->", "requests + BeautifulSoup cukup" if ada_di_html else "kemungkinan dirender JavaScript")

# Metadata scraping (verifikasi ulang hasil)
meta_scraping = {
    "berkas_html": "data/raw/halaman_wikipedia.html",
    "berkas_tabel": "data/processed/tabel_wikipedia_kota.csv",
    "situs_sumber": URL_HALAMAN,
    "waktu_scraping_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "user_agent": USER_AGENT,
    "jeda_antar_request_detik": DELAY,
    "timeout_detik": TIMEOUT,
    "jumlah_record": int(len(df_scrape)),
    "robots_txt_mengizinkan": hasil_robots["boleh_diakses"],
    "bukti_robots": "data/raw/robots_txt_evidence.txt",
    "lisensi_konten": "CC BY-SA 4.0 (Wikipedia), atribusi wajib",
}
Path("docs/scraping_meta.json").write_text(json.dumps(meta_scraping, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(meta_scraping, indent=2, ensure_ascii=False))

Konten ada di HTML awal: True -> requests + BeautifulSoup cukup
{
  "berkas_html": "data/raw/halaman_wikipedia.html",
  "berkas_tabel": "data/processed/tabel_wikipedia_kota.csv",
  "situs_sumber": "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population",
  "waktu_scraping_utc": "2026-11-18T10:12:41+00:00",
  "user_agent": "INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawan@gmail.com)",
  "jeda_antar_request_detik": 1.5,
  "timeout_detik": 20,
  "jumlah_record": 95,
  "robots_txt_mengizinkan": true,
  "bukti_robots": "data/raw/robots_txt_evidence.txt",
  "lisensi_konten": "CC BY-SA 4.0 (Wikipedia), atribusi wajib"
}


In [85]:
# Salinan kerja; data mentah tidak diubah. Buang dua kolom artefak ekspor.
df_a = df_file.drop(columns=kolom_sisa).copy()
df_a["City"] = df_a["City"].str.strip()

# Agregasi cuaca harian -> satu baris per kota (kunci kanan unik)
cuaca_kota = (df_api.groupby("lokasi", as_index=False)
              .agg(suhu_rata2_c=("temperature_2m_mean", "mean"),
                   curah_hujan_total_mm=("precipitation_sum", "sum"),
                   durasi_sinar_matahari_rata2_detik=("sunshine_duration", "mean"))
              .rename(columns={"lokasi": "City"}))
wilayah = df_scrape.drop_duplicates("City")

print("Kunci unik (cuaca):", cuaca_kota["City"].is_unique, "| (wilayah):", wilayah["City"].is_unique)
print("Kota di dataset tanpa pasangan di cuaca :", set(df_a["City"]) - set(cuaca_kota["City"]))
print("Kota di dataset tanpa pasangan di wilayah:", set(df_a["City"]) - set(wilayah["City"]))

Kunci unik (cuaca): True | (wilayah): True
Kota di dataset tanpa pasangan di cuaca : set()
Kota di dataset tanpa pasangan di wilayah: set()


In [86]:
# Merge bertahap dengan validate="many_to_one": kiri boleh berulang, kanan harus unik (jika tidak -> MergeError)
n_awal = len(df_a)
merged = (df_a
          .merge(cuaca_kota, on="City", how="left", validate="many_to_one", indicator="_m_cuaca")
          .merge(wilayah, on="City", how="left", validate="many_to_one", indicator="_m_wilayah"))

# Bukti tidak ada duplikasi baris dan semua baris punya pasangan
assert len(merged) == n_awal, "Jumlah baris berubah -> ada duplikasi!"
assert not merged["Place_Id"].duplicated().any(), "Place_Id duplikat"
assert (merged["_m_cuaca"] == "both").all() and (merged["_m_wilayah"] == "both").all(), "Ada baris tanpa pasangan"
print(f"Baris awal: {n_awal} | hasil merge: {len(merged)}")
print("Duplikat Place_Id:", int(merged["Place_Id"].duplicated().sum()), "| duplikat baris penuh:", int(merged.drop(columns=['_m_cuaca','_m_wilayah']).duplicated().sum()))
merged = merged.drop(columns=["_m_cuaca", "_m_wilayah"])
merged.head()

Baris awal: 437 | hasil merge: 437
Duplikat Place_Id: 0 | duplikat baris penuh: 0


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,suhu_rata2_c,curah_hujan_total_mm,durasi_sinar_matahari_rata2_detik,Province,Region,Population_2020
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088


In [87]:
# Simpan hasil gabungan beserta metadata
OUT = "data/processed/merged_dataset.csv"
merged.to_csv(OUT, index=False)
metadata = {
    "judul": TOPIK,
    "dibuat_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "email_kontak": EMAIL,
    "sumber": [p["sumber"] for p in PROVENANCE],
    "jumlah_baris_berkas": int(n_awal),
    "jumlah_baris_api": int(len(df_api)),
    "jumlah_baris_scraping": int(len(df_scrape)),
    "jumlah_baris_hasil": int(len(merged)),
    "jumlah_kolom_hasil": int(merged.shape[1]),
    "kolom": list(merged.columns),
    "kunci_merge": ["City"],
    "validate": "many_to_one",
    "metode_merge": "left",
    "duplikasi_place_id": int(merged["Place_Id"].duplicated().sum()),
    "kolom_dibuang": kolom_sisa,
    "satuan": {"Price": "rupiah", "Time_Minutes": "menit", "suhu_rata2_c": "derajat Celsius",
               "curah_hujan_total_mm": "mm per tahun 2024", "durasi_sinar_matahari_rata2_detik": "detik per hari"},
    "file_hasil": OUT,
    "md5_hasil": md5_file(OUT),
}
Path("data/processed/merged_dataset.metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metadata, indent=2, ensure_ascii=False)[:1800])

{
  "judul": "Destinasi Wisata Indonesia",
  "dibuat_utc": "2026-11-18T10:12:41+00:00",
  "email_kontak": "wiraastinarasmawan@gmail.com",
  "sumber": [
    "Kaggle - Indonesia Tourism Destination",
    "Open-Meteo Historical API - Jakarta",
    "Open-Meteo Historical API - Yogyakarta",
    "Open-Meteo Historical API - Bandung",
    "Open-Meteo Historical API - Semarang",
    "Open-Meteo Historical API - Surabaya",
    "Wikipedia robots.txt",
    "Wikipedia - List of Indonesian cities by population"
  ],
  "jumlah_baris_berkas": 437,
  "jumlah_baris_api": 1830,
  "jumlah_baris_scraping": 95,
  "jumlah_baris_hasil": 437,
  "jumlah_kolom_hasil": 17,
  "kolom": [
    "Place_Id",
    "Place_Name",
    "Description",
    "Category",
    "City",
    "Price",
    "Rating",
    "Time_Minutes",
    "Coordinate",
    "Lat",
    "Long",
    "suhu_rata2_c",
    "curah_hujan_total_mm",
    "durasi_sinar_matahari_rata2_detik",
    "Province",
    "Region",
    "Population_2020"
  ],
  "kunci_merge": 

In [88]:

# ==================================================
# DATA ETHICS CHECKLIST - 9 BUTIR
# ==================================================

# Verifikasi lisensi dan asal data
lisensi_ok = False

asal_ok = (
    ASAL_DATA_KAGGLE.strip() != ""
    and "ISI KETERANGAN" not in ASAL_DATA_KAGGLE.upper()
    and "TIDAK DIJELASKAN" not in ASAL_DATA_KAGGLE.upper()
)

status_asal = (
    "Ya" if ASAL_BUKAN_PLATFORM_TERLARANG is True
    else "Tidak" if ASAL_BUKAN_PLATFORM_TERLARANG is False
    else "Belum dipastikan"
)

checklist = pd.DataFrame([
    [
        1,
        "Tujuan pengumpulan jelas dan relevan (akademik, sektor pariwisata)",
        "Ya",
        "UTS Data Acquisition & Management; topik destinasi wisata Indonesia dan cuaca"
    ],
    [
        2,
        "Lisensi dataset jelas dan mengizinkan penggunaan akademik",
        "Ya" if lisensi_ok else "Perlu verifikasi",
        f"Lisensi Kaggle: {LISENSI_KAGGLE}. "
        "Open-Meteo: CC BY 4.0. Wikipedia: CC BY-SA 4.0. "
        f"Data Card: {FILE_URL}"
    ],
    [
        3,
        "Ketentuan portal sumber dibaca dan dipatuhi; asal data bukan platform terlarang",
        "Ya" if (
            SUDAH_BACA_TOS is True
            and asal_ok
            and ASAL_BUKAN_PLATFORM_TERLARANG is True
        ) else "Perlu verifikasi",
        f"ToS: {URL_KAGGLE_TOS}. "
        f"Asal data: {ASAL_DATA_KAGGLE}. "
        f"Bukan platform terlarang: {status_asal}"
    ],
    [
        4,
        "Tidak ada PII langsung (nama orang, NIK, email, telepon, alamat pribadi)",
        "Ya" if len(PII_LANGSUNG) == 0 else "Perlu ditinjau",
        f"Pemindaian nama kolom dan pola nilai: "
        f"{len(PII_LANGSUNG)} kandidat PII langsung. "
        "Data berisi tempat wisata, bukan individu"
    ],
    [
        5,
        "Risiko quasi-identifier / re-identifikasi dinilai",
        "Ya",
        f"Quasi-identifier lokasi: {', '.join(QUASI)}. "
        "Data agregat tempat wisata, risiko re-identifikasi rendah"
    ],
    [
        6,
        "robots.txt diperiksa dan halaman yang di-scrape diizinkan",
        "Ya" if hasil_robots["boleh_diakses"] else "Tidak",
        f"{hasil_robots['robots_url']} "
        f"(status {hasil_robots['status_code_robots']}); "
        f"can_fetch={hasil_robots['boleh_diakses']}; "
        "bukti data/raw/robots_txt_evidence.txt dan docs/robots_check.json"
    ],
    [
        7,
        "Timeout, User-Agent berisi email kampus, dan jeda antar-request dipakai",
        "Ya",
        f"UA: {USER_AGENT}; timeout {TIMEOUT}s; "
        f"jeda {DELAY}s; retry dengan backoff di polite_get"
    ],
    [
        8,
        "API key/token tidak ditulis di notebook",
        "Ya",
        "Open-Meteo tanpa API key; kredensial Kaggle API (jika dipakai) dari Colab Secrets"
    ],
    [
        9,
        "Atribusi sumber dan provenance (URL, tanggal akses, lisensi, MD5) dicatat",
        "Ya",
        f"docs/provenance.csv ({len(PROVENANCE)} sumber); "
        "atribusi Open-Meteo dan Wikipedia dicantumkan"
    ],
], columns=["No", "Butir Etika", "Status", "Bukti"])

# Simpan hasil checklist
checklist.to_csv(
    "data/processed/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

display(checklist)


,No,Butir Etika,Status,Bukti
0,1,Tujuan pengumpulan jelas dan relevan (akademik...,Ya,UTS Data Acquisition & Management; topik desti...
1,2,Lisensi dataset jelas dan mengizinkan pengguna...,Perlu verifikasi,Lisensi Kaggle: Data files © Original Authors....
2,3,Ketentuan portal sumber dibaca dan dipatuhi; a...,Perlu verifikasi,ToS: https://www.kaggle.com/terms. Asal data: ...
3,4,"Tidak ada PII langsung (nama orang, NIK, email...",Ya,Pemindaian nama kolom dan pola nilai: 0 kandid...
4,5,Risiko quasi-identifier / re-identifikasi dinilai,Ya,"Quasi-identifier lokasi: City, Coordinate, Lat..."
5,6,robots.txt diperiksa dan halaman yang di-scrap...,Ya,https://en.wikipedia.org/robots.txt (status 20...
6,7,"Timeout, User-Agent berisi email kampus, dan j...",Ya,UA: INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawa...
7,8,API key/token tidak ditulis di notebook,Ya,Open-Meteo tanpa API key; kredensial Kaggle AP...
8,9,"Atribusi sumber dan provenance (URL, tanggal a...",Ya,docs/provenance.csv (8 sumber); atribusi Open-...


In [89]:
# Simpan provenance seluruh sumber dan verifikasi bahwa data/raw tidak berubah (MD5 dihitung ulang)
prov = pd.DataFrame(PROVENANCE)
prov.to_csv("docs/provenance.csv", index=False)
display(prov[["sumber", "metode_akuisisi", "tanggal_akses_utc", "lisensi", "md5"]])
for p in PROVENANCE:
    assert md5_file(p["berkas_mentah"]) == p["md5"], f"Berkas mentah berubah: {p['berkas_mentah']}"
print("✔ Semua berkas mentah utuh (MD5 cocok)")

,sumber,metode_akuisisi,tanggal_akses_utc,lisensi,md5
0,Kaggle - Indonesia Tourism Destination,unduh manual dari Kaggle,2026-11-18 10:12:26,Data files © Original Authors. Pada bagian Ack...,722ca7f02aa2f847c41798c3f6a5d7a5
1,Open-Meteo Historical API - Jakarta,API,2026-11-18 10:12:28,CC BY 4.0,5947251f4e5024368641b3a3efa6fb64
2,Open-Meteo Historical API - Yogyakarta,API,2026-11-18 10:12:30,CC BY 4.0,564017edb8037adb5cb76e4ec8c630a5
3,Open-Meteo Historical API - Bandung,API,2026-11-18 10:12:32,CC BY 4.0,ff7469b63c5992f823ed7b2fd51b89a7
4,Open-Meteo Historical API - Semarang,API,2026-11-18 10:12:34,CC BY 4.0,0daa501d04685d256c28d552d7732480
5,Open-Meteo Historical API - Surabaya,API,2026-11-18 10:12:36,CC BY 4.0,071270d667cfb62c6240a7313b38bd15
6,Wikipedia robots.txt,HTTP GET,2026-11-18 10:12:38,n/a (bukti pemeriksaan kepatuhan),fc15d80fd6a4afd9f22975e740c1b48c
7,Wikipedia - List of Indonesian cities by popul...,web scraping statis,2026-11-18 10:12:40,CC BY-SA 4.0,7024fa5072368502909e684c79cf42ca


✔ Semua berkas mentah utuh (MD5 cocok)


In [90]:
# Cek kepatuhan terhadap ketentuan UTS (otomatis)
cek = [
 ("Dataset berkas >= 100 baris (Soal 1)", n_baris >= 100),
 ("Estimasi storage dihitung (Soal 1)", ESTIMASI_STORAGE_MB > 0),
 ("Lisensi Kaggle terisi dari Data Card (Soal 2)", lisensi_ok),
 ("Asal data Kaggle diperiksa dan bukan platform terlarang (Soal 2)", ASAL_BUKAN_PLATFORM_TERLARANG is True),
 ("Checklist 9 butir tanpa butir 'BELUM' (Soal 2)", not checklist["Status"].str.contains("BELUM|TIDAK|Perlu").any()),
 ("API menghasilkan >= 30 baris (Soal 3)", len(df_api) >= 30),
 ("robots.txt mengizinkan halaman scraping (Soal 4)", bool(hasil_robots["boleh_diakses"])),
 ("Tabel scraping tersimpan (Soal 4)", os.path.exists("data/processed/tabel_wikipedia_kota.csv")),
 ("Merge: baris tidak bertambah, tidak ada duplikat (Soal 5)", len(merged) == n_awal and not merged["Place_Id"].duplicated().any()),
 ("User-Agent memuat email kampus", EMAIL.endswith(".ac.id") and EMAIL in USER_AGENT),
 ("Semua sumber punya provenance + MD5", len(PROVENANCE) >= 8 and all(p["md5"] for p in PROVENANCE)),
]
for nama, ok in cek:
    print(("✔ " if ok else "✗ ") + nama)

✔ Dataset berkas >= 100 baris (Soal 1)
✔ Estimasi storage dihitung (Soal 1)
✗ Lisensi Kaggle terisi dari Data Card (Soal 2)
✗ Asal data Kaggle diperiksa dan bukan platform terlarang (Soal 2)
✗ Checklist 9 butir tanpa butir 'BELUM' (Soal 2)
✔ API menghasilkan >= 30 baris (Soal 3)
✔ robots.txt mengizinkan halaman scraping (Soal 4)
✔ Tabel scraping tersimpan (Soal 4)
✔ Merge: baris tidak bertambah, tidak ada duplikat (Soal 5)
✗ User-Agent memuat email kampus
✔ Semua sumber punya provenance + MD5


In [92]:
# Kemas data dan docs menjadi zip lalu unduh (untuk diunggah ke GitHub/Colab bersama notebook ini)
!zip -qr UTS_DAM_Indonesia_Tourism.zip data docs
from google.colab import files
files.download("UTS_DAM_Indonesia_Tourism.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>